In [6]:

from pathlib import Path
import pandas as pd
import json

# try to use tiktoken for accurate token cropping, else fallback to whitespace-based crop
try:
    import tiktoken
except Exception:
    tiktoken = None

def crop_to_n_tokens(text: str, n: int = 512, encoding_name: str = "cl100k_base") -> str:
    if text is None:
        return ""
    try:
        if tiktoken is None:
            raise RuntimeError("tiktoken not available")
        enc = tiktoken.get_encoding(encoding_name)
        toks = enc.encode(text)
        if len(toks) <= n:
            return text
        return enc.decode(toks[:n])
    except Exception:
        parts = str(text).split()
        return " ".join(parts[:n]) if len(parts) > n else text

LANG_MAP = {
    "english": ("question", "intital_malicious_english", "sample_200_english.json"),
    "hindi": ("hindi", "intital_malicious_hindi", "sample_200_hindi.json"),
    "bengali": ("bengali", "intital_malicious_bengali", "sample_200_bengali.json"),
}

def build_records(df: pd.DataFrame, q_col: str, r_col: str, token_limit: int) -> list:
    records = []
    for _, row in df.iterrows():
        q = str(row.get(q_col, "")).strip()
        r = str(row.get(r_col, "")).strip()
        if not q and not r:
            continue
        r_crop = crop_to_n_tokens(r, n=token_limit)
        instruction = (q + " " + r_crop).strip() if q else r_crop
        records.append({
            "instruction": instruction,
            "instruction_translated": q,
            "category": row.get("category"),
        })
    return records

# PARAMETERS — adjust as needed
csv_path = Path("initial_malicious_final.csv")
outdir = Path(".")
sample_n = 200
token_limit = 512
seed = 42

df = pd.read_csv(csv_path)
if sample_n and sample_n < len(df):
    sample_df = df.sample(n=sample_n, random_state=seed).reset_index(drop=True)
else:
    sample_df = df.copy()

outdir.mkdir(parents=True, exist_ok=True)

for lang, (q_col, r_col, out_name) in LANG_MAP.items():
    if q_col not in sample_df.columns or r_col not in sample_df.columns:
        print(f"[WARN] Missing columns for {lang}: {q_col} or {r_col} not found — skipping.")
        continue
    records = build_records(sample_df, q_col, r_col, token_limit)
    out_path = outdir / out_name
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"[OK] Wrote {len(records)} records -> {out_path}")


[OK] Wrote 200 records -> sample_200_english.json
[OK] Wrote 200 records -> sample_200_hindi.json
[OK] Wrote 200 records -> sample_200_bengali.json
